# SIB - Exercício 8: RidgeRegression com Least Squares

Em vez de iterar com gradient descent, a solução dos mínimos quadrados resolve o problema
analiticamente, numa única operação:

$$\theta = \left( X^T X + \lambda \begin{bmatrix} 0 & & & \\ & 1 & & \\ & & \ddots & \\ & & & 1 \end{bmatrix} \right)^{-1} X^T y$$

A primeira posição da matriz de penalização é 0 para que o intercept (`theta_zero`) **não** seja penalizado.

In [1]:
import numpy as np
import matplotlib.pyplot as plt

from si.data.dataset import Dataset
from si.io.csv_file import read_csv
from si.model_selection.split import train_test_split
from si.models.linear_regression import RidgeRegression
from si.models.ridge_regression_least_squares import RidgeRegressionLeastSquares

## 1. Dataset e divisão treino/teste

In [2]:
cpu = read_csv('../datasets/cpu/cpu.csv', sep=',', features=True, label=True)
train, test = train_test_split(cpu, test_size=0.2, random_state=42)

print('treino:', train.shape(), '| teste:', test.shape())

treino: (168, 6) | teste: (41, 6)


## 2. Treino

In [3]:
ls_model = RidgeRegressionLeastSquares(l2_penalty=1.0, scale=True)
ls_model.fit(train)

print('theta_zero: {:.4f}'.format(ls_model.theta_zero))
for feature, coef in zip(train.features, ls_model.theta):
    print('  {:<6s} {:>9.4f}'.format(feature, coef))

print()
print('mse (treino): {:.4f}'.format(ls_model.score(train)))
print('mse (teste):  {:.4f}'.format(ls_model.score(test)))

theta_zero: 99.0595
  syct     11.2725
  mmin     36.9703
  mmax     64.0489
  cach     25.7607
  chmin     2.7745
  chmax    32.0951

mse (treino): 3102.2550
mse (teste):  5806.0785


## 3. Validação: comparação com o gradient descent

Se as duas implementações estiverem corretas, o gradient descent tem de **convergir para a mesma
solução** do least squares (com iterações suficientes). É o melhor teste de correção que temos aqui.

In [4]:
gd_model = RidgeRegression(l2_penalty=1.0, alpha=0.01, max_iter=100000, patience=1000, scale=True).fit(train)

print('{:<8s} {:>14s} {:>14s} {:>12s}'.format('', 'least squares', 'grad. descent', 'diferença'))
print('{:<8s} {:>14.4f} {:>14.4f} {:>12.6f}'.format(
    'theta_0', ls_model.theta_zero, gd_model.theta_zero, abs(ls_model.theta_zero - gd_model.theta_zero)))
for feature, a, b in zip(train.features, ls_model.theta, gd_model.theta):
    print('{:<8s} {:>14.4f} {:>14.4f} {:>12.6f}'.format(feature, a, b, abs(a - b)))

print()
print('iterações do gradient descent:', len(gd_model.cost_history))
print('mse teste  ->  least squares: {:.4f} | gradient descent: {:.4f}'.format(
    ls_model.score(test), gd_model.score(test)))

          least squares  grad. descent    diferença
theta_0         99.0595        99.0595     0.000000
syct            11.2725        11.2718     0.000640
mmin            36.9703        37.0167     0.046439
mmax            64.0489        64.0070     0.041849
cach            25.7607        25.7504     0.010282
chmin            2.7745         2.7634     0.011111
chmax           32.0951        32.1165     0.021485

iterações do gradient descent: 2430
mse teste  ->  least squares: 5806.0785 | gradient descent: 5803.3386


## 4. Validação: solução fechada calculada à mão

Caso pequeno, sem escalamento, para confirmar a álgebra passo a passo.

In [5]:
X = np.array([[1.0, 2.0], [3.0, 4.0], [5.0, 6.0], [7.0, 9.0]])
y = np.array([1.0, 2.0, 3.0, 4.0])
toy = Dataset(X=X, y=y)

toy_model = RidgeRegressionLeastSquares(l2_penalty=2.0, scale=False).fit(toy)

# cálculo manual
X_intercept = np.c_[np.ones(4), X]
penalty = 2.0 * np.eye(3)
penalty[0, 0] = 0
expected = np.linalg.inv(X_intercept.T.dot(X_intercept) + penalty).dot(X_intercept.T).dot(y)

print('esperado:', np.round(expected, 6))
print('obtido:  ', np.round(np.r_[toy_model.theta_zero, toy_model.theta], 6))
print('coincide:', np.allclose(expected, np.r_[toy_model.theta_zero, toy_model.theta]))

esperado: [0.444444 0.222222 0.222222]
obtido:   [0.444444 0.222222 0.222222]
coincide: True


## 5. O intercept não é penalizado

Com X centrado, `theta_zero` tem de ser sempre a média de y, independentemente do lambda.

In [6]:
X_centered = np.array([[-1.0], [0.0], [1.0]])
y_simple = np.array([2.0, 4.0, 6.0])
centered = Dataset(X=X_centered, y=y_simple)

print('média de y:', y_simple.mean())
for l2 in [0.0, 1.0, 100.0, 10000.0]:
    m = RidgeRegressionLeastSquares(l2_penalty=l2, scale=False).fit(centered)
    print('  lambda = {:>8.0f}  ->  theta_zero = {:.6f} | theta = {:.6f}'.format(
        l2, m.theta_zero, m.theta[0]))

média de y: 4.0
  lambda =        0  ->  theta_zero = 4.000000 | theta = 2.000000
  lambda =        1  ->  theta_zero = 4.000000 | theta = 1.333333
  lambda =      100  ->  theta_zero = 4.000000 | theta = 0.039216
  lambda =    10000  ->  theta_zero = 4.000000 | theta = 0.000400


## 6. Efeito do lambda nos coeficientes

In [7]:
lambdas = np.logspace(-2, 4, 40)
paths = np.array([RidgeRegressionLeastSquares(l2_penalty=l2, scale=True).fit(train).theta for l2 in lambdas])

plt.figure(figsize=(7, 4))
for j, feature in enumerate(train.features):
    plt.plot(lambdas, paths[:, j], label=feature)
plt.xscale('log')
plt.xlabel('l2_penalty (lambda)')
plt.ylabel('theta')
plt.title('Encolhimento dos coeficientes')
plt.legend()
plt.tight_layout()
plt.show()

## Conclusões

- As duas implementações chegam à mesma solução, o que valida ambas.
- O least squares é exato e instantâneo neste dataset (6 features); o gradient descent precisa
  de milhares de iterações para lá chegar.
- A vantagem do gradient descent aparece quando n é grande: inverter uma matriz n×n custa O(n³)
  e pode ser numericamente instável.